In [ ]:
# Install biobouncer (safe to re-run, a no-op if it is already installed).
%pip install -q biobouncer pandas narwhals

# biobouncer (Python): a gate for biological inputs

**biobouncer** validates biological identifiers (gene symbols, ontology terms,
variant formats, database accessions) through one small API, with the *same*
answers in Python and R.

Four validation modes:

| mode | what it checks | network |
|------|----------------|---------|
| `pattern`   | is the string well-formed? (regex/grammar) | offline |
| `cache`     | does the id exist in a pinned snapshot?    | offline |
| `remote`    | is the id live in the source API?          | online  |
| `existence` | snapshot first, then remote                | either  |

Everything below runs **offline** (pattern + cache) so it is safe to record.

In [1]:
import pandas as pd
import biobouncer as bb


def to_df(results):
    "Turn a list of Result records into a tidy DataFrame."
    return pd.DataFrame([r.__dict__ for r in results])


print("biobouncer", bb.__version__)

biobouncer 0.1.2


## 1. Discover: what can biobouncer check?

In [2]:
srcs = bb.sources()
print(f"{len(srcs)} sources supported\n")
print(srcs)

46 sources supported

['bto', 'cdd', 'chebi', 'chembl', 'cl', 'clinvar', 'complexportal', 'cosmic', 'dbsnp', 'doid', 'drugbank', 'ec', 'eco', 'efo', 'ensembl', 'go', 'hgnc', 'hgvs', 'hp', 'inchikey', 'interpro', 'mirbase', 'mirbase_hairpin', 'mondo', 'mp', 'ncbifam', 'ncbitaxon', 'ncit', 'opentargets', 'orphanet', 'panther', 'pato', 'pdb', 'pfam', 'pharmgkb', 'prints', 'prosite', 'reactome', 'refseq', 'rfam', 'smart', 'so', 'uberon', 'uniparc', 'uniprot', 'wikipathways']


In [3]:
# Rich metadata for every source: id example, supported modes, species/version aware
info = pd.DataFrame(bb.source_info())
info

,key,name,example,modes,species_aware,version_aware
0,bto,BRENDA Tissue Ontology,BTO:0000759,"[pattern, cache, remote]",False,True
1,cdd,CDD,cd00029,"[pattern, remote]",False,False
2,chebi,ChEBI,CHEBI:15377,"[pattern, cache, remote]",False,True
3,chembl,ChEMBL,CHEMBL25,"[pattern, remote]",False,False
4,cl,Cell Ontology,CL:0000236,"[pattern, cache, remote]",False,True
5,clinvar,ClinVar,VCV000012345,"[pattern, remote]",False,False
6,complexportal,Complex Portal,CPX-2158,"[pattern, remote]",False,False
7,cosmic,COSMIC,COSM476,[pattern],False,False
8,dbsnp,dbSNP,rs7412,"[pattern, remote]",False,False
9,doid,Human Disease Ontology,DOID:9352,"[pattern, cache, remote]",False,True


## 2. Snapshots: the bundled offline data

In [4]:
print("cache dir:", bb.cache_dir(), "\n")
pd.DataFrame(bb.snapshots())

cache dir: C:\Users\Samuel\AppData\Local\biobouncer\biobouncer\Cache 



,source,version,n_ids,location
0,bto,sample,7,bundled
1,chebi,sample,5,bundled
2,cl,sample,7,bundled
3,doid,sample,7,bundled
4,efo,sample,5,bundled
5,go,sample,6,bundled
6,hgnc,2026-07-07,45019,bundled
7,hgnc,sample,7,bundled
8,hp,sample,7,bundled
9,mondo,sample,6,bundled


## 3. Mode `pattern`: is the string well-formed? (offline)

Format-only. Note the normalization + repair *suggestion* for a badly-cased id,
and that a well-formed GO id is correctly rejected when checked as MONDO.

In [5]:
to_df(bb.check_id(["MONDO:0005148", "mondo:5148", "GO:0006915"], source_db="mondo"))

,input,valid,normalized,suggestion,source_db,version,species,how,error
0,MONDO:0005148,True,MONDO:0005148,NaN,mondo,None,None,pattern,None
1,mondo:5148,False,NaN,MONDO:0005148,mondo,None,None,pattern,None
2,GO:0006915,False,NaN,NaN,mondo,None,None,pattern,None


## 4. Mode `cache`: does the id actually exist? (offline snapshot)

Here we load a **real, messy** gene-disease association table and validate every
column. `cache` mode catches ids that are well-formed but wrong, and proposes the
current id for renamed/obsolete entries.

In [6]:
df = pd.read_csv("data/associations.csv")
df

,gene,disease,process
0,TP53,MONDO:0005148,GO:0006915
1,MLL,MONDO:0004975,GO:0006917
2,CARS,MONDO:0007739,GO:0003674
3,BRCA2,mondo:5148,GO:0005515
4,EGFR,MONDO:0005147,GO:0008150
5,notagene,MONDO:9999999,GO:9999999
6,NaN,MONDO:0018076,NaN


In [7]:
# One report per column. Each summarises valid / repairable / invalid / missing.
for col, source in [("gene", "hgnc"), ("disease", "mondo"), ("process", "go")]:
    print(bb.report(df[col], source, how="cache"))

<biobouncer report on 'hgnc' (cache mode): 3 valid, 2 repairable, 1 invalid, 1 missing of 7>
<biobouncer report on 'mondo' (cache mode): 5 valid, 1 repairable, 1 invalid, 0 missing of 7>
<biobouncer report on 'go' (cache mode): 4 valid, 1 repairable, 1 invalid, 1 missing of 7>


In [8]:
# Look inside one report: MLL -> KMT2A, CARS -> CARS1 are auto-suggested.
to_df(bb.check_id(df["gene"].tolist(), source_db="hgnc", how="cache"))

,input,valid,normalized,suggestion,source_db,version,species,how,error
0,TP53,True,TP53,NaN,hgnc,2026-07-07,None,cache,None
1,MLL,False,NaN,KMT2A,hgnc,2026-07-07,None,cache,None
2,CARS,False,NaN,CARS1,hgnc,2026-07-07,None,cache,None
3,BRCA2,True,BRCA2,NaN,hgnc,2026-07-07,None,cache,None
4,EGFR,True,EGFR,NaN,hgnc,2026-07-07,None,cache,None
5,notagene,False,NaN,NaN,hgnc,2026-07-07,None,cache,None
6,NaN,None,NaN,NaN,hgnc,2026-07-07,None,cache,None


### Repair the whole table in one pass

In [9]:
clean = df.copy()
clean["gene"] = bb.report(df["gene"], "hgnc", how="cache").repair()
clean["disease"] = bb.report(df["disease"], "mondo", how="cache").repair()
clean["process"] = bb.report(df["process"], "go", how="cache").repair()
clean

,gene,disease,process
0,TP53,MONDO:0005148,GO:0006915
1,KMT2A,MONDO:0004975,GO:0006915
2,CARS1,MONDO:0007739,GO:0003674
3,BRCA2,MONDO:0005148,GO:0005515
4,EGFR,MONDO:0005147,GO:0008150
5,notagene,MONDO:9999999,GO:9999999
6,NaN,MONDO:0018076,NaN


## 5. `pattern` across many sources at once

A second real dataset mixes accessions from different databases. We validate each
row against its own `source_db`: UniProt, Ensembl, RefSeq, dbSNP, HGVS, ChEBI.

In [10]:
ids = pd.read_csv("data/identifiers.csv")

rows = []
for source, grp in ids.groupby("source_db"):
    for r in bb.check_id(grp["value"].tolist(), source_db=source, how="pattern"):
        rows.append(r.__dict__)
pd.DataFrame(rows)[["input", "source_db", "valid", "normalized", "suggestion"]]

,input,source_db,valid,normalized,suggestion
0,CHEBI:15377,chebi,True,CHEBI:15377,None
1,CHEBI:99999999,chebi,True,CHEBI:99999999,None
2,rs7412,dbsnp,True,rs7412,None
3,12345,dbsnp,False,NaN,None
4,ENSG00000139618,ensembl,True,ENSG00000139618,None
5,ENSG123,ensembl,False,NaN,None
6,NM_004006.2:c.4375C>T,hgvs,True,NM_004006.2:c.4375C>T,None
7,c.bad,hgvs,False,NaN,None
8,NM_000546.6,refseq,True,NM_000546.6,None
9,NM_bad,refseq,False,NaN,None


## 6. Mode `remote`: live check against the source API (optional)

Needs internet, so it is wrapped to fail gracefully during an offline recording.
`remote` is also species-aware for sources like Ensembl.

In [11]:
try:
    out = bb.check_id(
        "ENSG00000139618", source_db="ensembl", how="remote", species="homo_sapiens"
    )
    print(to_df(out))
except Exception as e:
    print("remote skipped (offline):", type(e).__name__, e)

             input  valid       normalized suggestion source_db  \
0  ENSG00000139618   True  ENSG00000139618       None   ensembl   

                version       species     how error  
0  2026-07-14T08:00:09Z  homo_sapiens  remote  None  


## 7. Drops into your validation stack

biobouncer ships adapters so the same checks power schema validation:

```python
import pandera.pandas as pa
from biobouncer.checks import is_id

schema = pa.DataFrameSchema({
    "disease": pa.Column(str, is_id(source_db="mondo", how="cache")),
    "gene":    pa.Column(str, is_id(source_db="hgnc",  how="cache")),
})
```

```python
from pydantic import BaseModel
from biobouncer.types import Id

class Assoc(BaseModel):
    disease: Id("mondo", how="pattern")
    gene:    Id("hgnc",  how="cache")
```

One gate for biological inputs: offline by default, identical answers in R.